In [ ]:
import numpy as np
from scipy.stats import norm

class ObjectTracker:
    def __init__(self, process_noise=0.1, measurement_noise=0.1, initial_state_uncertainty=1.0):
        self.process_noise = process_noise
        self.measurement_noise = measurement_noise
        
        # State: [position, velocity]
        self.state = np.zeros((2, 1))
        self.P = np.eye(2) * initial_state_uncertainty
        
        self.Q = np.array([[0.25*process_noise**4, 0.5*process_noise**3],
                           [0.5*process_noise**3, process_noise**2]])
        self.R = measurement_noise**2
        
        self.H = np.array([[1, 0]])
        self.F = np.array([[1, 1],
                           [0, 1]])
        
        self.direction_probability = 0.5
        self.direction_confidence = 0.5
        
        self.change_detection_threshold = 3.0
        self.change_detection_window = 10
        self.measurements_history = []

    def update(self, measurement):
        # Prediction
        self.state = self.F @ self.state
        self.P = self.F @ self.P @ self.F.T + self.Q
        
        # Update
        y = measurement - self.H @ self.state
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T / S
        
        self.state = self.state + K * y
        self.P = (np.eye(2) - K @ self.H) @ self.P
        
        # Direction inference
        velocity = self.state[1]
        self.direction_probability = norm.cdf(velocity, 0, np.sqrt(self.P[1,1]))
        self.direction_confidence = 2 * abs(self.direction_probability - 0.5)
        
        # Change detection
        self.measurements_history.append(measurement)
        if len(self.measurements_history) > self.change_detection_window:
            self.measurements_history.pop(0)
            
        if len(self.measurements_history) == self.change_detection_window:
            recent_mean = np.mean(self.measurements_history[-(self.change_detection_window//2):])
            past_mean = np.mean(self.measurements_history[:self.change_detection_window//2])
            if abs(recent_mean - past_mean) > self.change_detection_threshold:
                self.reset_state()
        
        return self.get_direction(), self.direction_confidence, self.state[1]

    def get_direction(self):
        return "right" if self.direction_probability > 0.5 else "left"

    def reset_state(self):
        self.state[1] = 0  # Reset velocity
        self.P[1,1] = self.P[0,0]  # Increase velocity uncertainty

def simulate_object_movement(num_steps, time_step=1.0):
    position = 0
    velocity = 0
    measurements = []
    
    for _ in range(num_steps):
        # Update position and velocity
        acceleration = np.random.normal(0, 0.1)
        velocity += acceleration * time_step
        position += velocity * time_step + 0.5 * acceleration * time_step**2
        
        # Add measurement noise
        measurement = position + np.random.normal(0, 0.1)
        measurements.append(measurement)
        
        # Occasionally flip direction
        if np.random.random() < 0.02:
            velocity = -velocity
    
    return measurements

# Example usage
num_steps = 1000
measurements = simulate_object_movement(num_steps)

tracker = ObjectTracker()

for measurement in measurements:
    direction, confidence, velocity = tracker.update(measurement)
    print(f"{measurement:.2f}, {direction}, Confidence: {confidence[0]:.1%}, Velocity: {velocity[0]:.2f}")


Thank you for providing such a detailed problem description. Based on your requirements, I suggest using a Kalman Filter combined with a change detection algorithm to infer the direction of movement and velocity of the object, while also detecting sudden changes in direction. This approach will provide a probabilistic estimate of the object's state and allow for quick response to direction changes.

Here's a Python implementation that addresses your requirements:

```python
import numpy as np
from scipy.stats import norm

class ObjectTracker:
    def __init__(self, process_noise=0.1, measurement_noise=0.1, initial_state_uncertainty=1.0):
        self.process_noise = process_noise
        self.measurement_noise = measurement_noise
        
        # State: [position, velocity]
        self.state = np.zeros(2)
        self.P = np.eye(2) * initial_state_uncertainty
        
        self.Q = np.array([[0.25*process_noise**4, 0.5*process_noise**3],
                           [0.5*process_noise**3, process_noise**2]])
        self.R = measurement_noise**2
        
        self.H = np.array([[1, 0]])
        self.F = np.array([[1, 1],
                           [0, 1]])
        
        self.direction_probability = 0.5
        self.direction_confidence = 0.5
        
        self.change_detection_threshold = 3.0
        self.change_detection_window = 10
        self.measurements_history = []

    def update(self, measurement):
        # Prediction
        self.state = self.F @ self.state
        self.P = self.F @ self.P @ self.F.T + self.Q
        
        # Update
        y = measurement - self.H @ self.state
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T / S
        
        self.state += K * y
        self.P = (np.eye(2) - K @ self.H) @ self.P
        
        # Direction inference
        velocity = self.state[1]
        self.direction_probability = norm.cdf(velocity, 0, np.sqrt(self.P[1,1]))
        self.direction_confidence = 2 * abs(self.direction_probability - 0.5)
        
        # Change detection
        self.measurements_history.append(measurement)
        if len(self.measurements_history) > self.change_detection_window:
            self.measurements_history.pop(0)
            
        if len(self.measurements_history) == self.change_detection_window:
            recent_mean = np.mean(self.measurements_history[-(self.change_detection_window//2):])
            past_mean = np.mean(self.measurements_history[:self.change_detection_window//2])
            if abs(recent_mean - past_mean) > self.change_detection_threshold:
                self.reset_state()
        
        return self.get_direction(), self.direction_confidence, self.state[1]

    def get_direction(self):
        return "right" if self.direction_probability > 0.5 else "left"

    def reset_state(self):
        self.state[1] = 0  # Reset velocity
        self.P[1,1] = self.P[0,0]  # Increase velocity uncertainty

def simulate_object_movement(num_steps, time_step=1.0):
    position = 0
    velocity = 0
    measurements = []
    
    for _ in range(num_steps):
        # Update position and velocity
        acceleration = np.random.normal(0, 0.1)
        velocity += acceleration * time_step
        position += velocity * time_step + 0.5 * acceleration * time_step**2
        
        # Add measurement noise
        measurement = position + np.random.normal(0, 0.1)
        measurements.append(measurement)
        
        # Occasionally flip direction
        if np.random.random() < 0.02:
            velocity = -velocity
    
    return measurements

# Example usage
num_steps = 1000
measurements = simulate_object_movement(num_steps)

tracker = ObjectTracker()

for measurement in measurements:
    direction, confidence, velocity = tracker.update(measurement)
    print(f"Direction: {direction}, Confidence: {confidence:.2f}, Velocity: {velocity:.2f}")
```

This implementation includes the following key components:

1. **Kalman Filter**: Used to estimate the object's position and velocity based on the noisy measurements.

2. **Direction Inference**: The direction is inferred probabilistically based on the estimated velocity. The confidence of this inference is also calculated.

3. **Change Detection**: A simple change detection algorithm is implemented to detect sudden changes in direction. When a change is detected, the state estimation is reset to quickly adapt to the new direction.

4. **Simulation**: A simple object movement simulation is included for testing purposes.

The `ObjectTracker` class handles the Kalman filtering, direction inference, and change detection. The `update` method is called for each new measurement, returning the inferred direction, confidence, and estimated velocity.

To use this code:

1. Create an instance of `ObjectTracker`.
2. For each new measurement from your sensor, call the `update` method.
3. The method returns the inferred direction ("left" or "right"), the confidence of this inference (0 to 1), and the estimated velocity.

You can adjust the parameters in the `__init__` method to fine-tune the performance for your specific use case. The `process_noise` and `measurement_noise` parameters can be adjusted to match the characteristics of your system and sensor.

This solution provides a probabilistic inference of the object's direction, responds quickly to sudden changes in direction, and provides a measure of confidence in the inference. The estimated velocity is also available as a bonus.